In [ ]:
import gc
import glob
import re
import warnings
import zipfile
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import tifffile
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import convolve1d
from skimage.morphology import remove_small_objects
from tqdm.auto import tqdm

try:
    import cc3d
    USE_CC3D = True
except ImportError:
    USE_CC3D = False
    print("cc3d not found, will fall back to scipy ndimage.label")

from scipy import ndimage

warnings.filterwarnings('ignore')


# =============================================================================
# CONFIG
# =============================================================================
class Config:
    # Data
    DATA_ROOT = Path("/kaggle/input/vesuvius-challenge-surface-detection")
    TEST_IMAGES_DIR = DATA_ROOT / "test_images"
    TEST_CSV = DATA_ROOT / "test.csv"
    TRAIN_CSV = DATA_ROOT / "train.csv"
    TRAIN_IMAGES_DIR = DATA_ROOT / "train_images"
    TRAIN_LABELS_DIR = DATA_ROOT / "train_labels"

    # Output
    OUTPUT_DIR = Path("/kaggle/working/submission_masks")
    SUBMISSION_ZIP = Path("/kaggle/working/submission.zip")

    # Architecture (MUST match training)
    PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    FEATURES: List[int] = [32, 64, 128, 256, 320, 320]
    N_BLOCKS: List[int] = [1, 3, 4, 6, 6, 6]
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True

    # Inference
    OVERLAP: float = 0.7
    TTA_LEVEL: str = "flip"  # 'none', 'flip', 'full'
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True

    # Z smoothing (common boost)
    Z_SMOOTH: bool = True
    Z_KERNEL: List[float] = [0.25, 0.5, 0.25]

    # Post-processing
    FINAL_THRESHOLD: float = 0.55
    MIN_COMPONENT_SIZE: int = 25
    KEEP_LARGEST_K: Optional[int] = None  # e.g. 8, or None

    # -------------------------------------------------------------------------
    # CHECKPOINTS / ENSEMBLE
    # -------------------------------------------------------------------------
    # Base checkpoints to ensemble (keep this small for T4)
    CHECKPOINT_PATHS = [
        Path("/kaggle/input/vesuvius-challenge/pytorch/default/6/best_model.pth"),
    ]
    MODEL_WEIGHTS: Optional[List[float]] = None  # same length as CHECKPOINT_PATHS

    # Optional: build an averaged checkpoint from a glob and append it to ensemble
    BUILD_AVG_CHECKPOINT: bool = False
    CKPT_GLOB: str = "/kaggle/input/<your-dataset>/checkpoints/checkpoint_epoch_*.pth"
    AVG_LAST_N: int = 5
    AVG_OUT_PATH: Path = Path("/kaggle/working/avg_model.pth")

    # -------------------------------------------------------------------------
    # QUICK VALIDATION SWEEP (scroll-specific)
    # -------------------------------------------------------------------------
    RUN_VAL_SWEEP: bool = False
    TARGET_SCROLL_ID: int = 26002
    TEST_ID: str = "1407735"
    VAL_N: int = 10
    VAL_SEED: int = 42

    THRESH_SWEEP = [0.45, 0.50, 0.55, 0.60, 0.65]
    OVERLAP_SWEEP = [0.5, 0.7]
    TTA_SWEEP = ["none", "flip"]
    MIN_COMPONENT_SWEEP = [10, 25, 50, 100]
    KEEP_LARGEST_SWEEP = [None]  # optionally: [None, 8]


cfg = Config()
cfg.OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print(f"Device: {cfg.DEVICE}")
if cfg.DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")


# =============================================================================
# MODEL (must match training)
# =============================================================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, num_groups=8):
        super().__init__()
        num_groups = min(num_groups, out_ch)
        while out_ch % num_groups != 0 and num_groups > 1:
            num_groups -= 1
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(num_groups, out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(*[ConvBlock(channels, channels) for _ in range(n_convs)])

    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)

    def forward(self, x):
        b, c = x.shape[:2]
        cse = torch.sigmoid(self.cse_fc2(F.relu(self.cse_fc1(self.cse_pool(x).view(b, c))))).view(b, c, 1, 1, 1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    def __init__(
        self,
        in_ch=1,
        out_ch=1,
        features=None,
        n_blocks=None,
        use_scse=True,
        use_deep_supervision=True,
    ):
        super().__init__()
        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]

        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision

        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()

        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            encoder = nn.Sequential(ConvBlock(in_channels, feat), *[ResBlock(feat) for _ in range(nb)])
            self.encoders.append(encoder)
            self.attentions.append(scSEBlock(feat) if use_scse else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, 2))

        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        for i in range(len(features) - 2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i + 1], features[i], 2, 2))
            self.dec_convs.append(ConvBlock(features[i] * 2, features[i]))

        self.final = nn.Conv3d(features[0], out_ch, 1)

        if use_deep_supervision:
            self.ds_heads = nn.ModuleList()
            n_ds_outputs = min(3, len(features) - 1)
            for i in range(n_ds_outputs):
                ds_in_channels = features[-(i + 2)]
                self.ds_heads.append(nn.Conv3d(ds_in_channels, out_ch, 1))

    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)

        enc_features = enc_features[::-1]
        x = enc_features[0]

        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)

        return self.final(x)


# =============================================================================
# HELPERS
# =============================================================================

def compute_volume_stats(volume: np.ndarray):
    mean = float(volume.mean())
    std = float(volume.std()) + 1e-8
    return mean, std


def normalize_patch_with_stats(patch: np.ndarray, mean: float, std: float) -> np.ndarray:
    # Match training: volume-level zscore
    return (patch.astype(np.float32) - mean) / (std + 1e-8)


def sigmoid_np(x: np.ndarray) -> np.ndarray:
    # Stable sigmoid
    x = np.clip(x, -20, 20)
    return 1.0 / (1.0 + np.exp(-x))


def smooth_logits_z(logits: np.ndarray) -> np.ndarray:
    if not cfg.Z_SMOOTH:
        return logits
    k = np.asarray(cfg.Z_KERNEL, dtype=np.float32)
    k = k / (k.sum() + 1e-8)
    return convolve1d(logits, weights=k, axis=0, mode='nearest')


def connected_components_3d(volume: np.ndarray, connectivity=26):
    if USE_CC3D:
        return cc3d.connected_components(volume.astype(np.uint8), connectivity=connectivity)
    # scipy fallback
    if connectivity == 26:
        struct = ndimage.generate_binary_structure(3, 3)
    else:
        struct = ndimage.generate_binary_structure(3, 1)
    labeled, _ = ndimage.label(volume.astype(bool), structure=struct)
    return labeled


def keep_largest_components(binary: np.ndarray, k: int = 8) -> np.ndarray:
    labeled = connected_components_3d(binary, connectivity=26)
    if labeled.max() == 0:
        return binary.astype(np.uint8)
    counts = np.bincount(labeled.ravel())
    # ignore background 0
    if len(counts) <= 1:
        return binary.astype(np.uint8)
    comp_ids = np.argsort(counts[1:])[-k:] + 1
    return np.isin(labeled, comp_ids).astype(np.uint8)


def postprocess(prob: np.ndarray, threshold: float, min_component_size: int, keep_largest_k: Optional[int] = None) -> np.ndarray:
    binary = (prob > threshold)
    cleaned = remove_small_objects(binary, min_size=int(min_component_size), connectivity=3)
    out = cleaned.astype(np.uint8)
    if keep_largest_k is not None:
        out = keep_largest_components(out, k=int(keep_largest_k))
    return out


def dice_score(pred: np.ndarray, target: np.ndarray, ignore: Optional[np.ndarray] = None) -> float:
    pred = pred.astype(bool)
    tgt = target.astype(bool)
    if ignore is not None:
        ign = ignore.astype(bool)
        pred = pred & (~ign)
        tgt = tgt & (~ign)
    inter = (pred & tgt).sum()
    denom = pred.sum() + tgt.sum()
    return float((2 * inter + 1e-5) / (denom + 1e-5))


# =============================================================================
# SLIDING WINDOW INFERENCE (LOGITS)
# =============================================================================
@torch.no_grad()
def sliding_window_inference_logits(
    model,
    volume: np.ndarray,
    patch_size=(192, 192, 192),
    overlap=0.5,
    device="cuda",
    use_amp=True,
    vol_mean=None,
    vol_std=None,
) -> np.ndarray:
    model.eval().to(device)

    if vol_mean is None or vol_std is None:
        vol_mean, vol_std = compute_volume_stats(volume)

    D, H, W = volume.shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd * (1 - overlap)), int(ph * (1 - overlap)), int(pw * (1 - overlap))

    logit_sum = np.zeros((D, H, W), dtype=np.float32)
    count = np.zeros((D, H, W), dtype=np.float32)

    gz = np.exp(-0.5 * ((np.arange(pd) - pd / 2) / (pd * 0.125)) ** 2)
    gy = np.exp(-0.5 * ((np.arange(ph) - ph / 2) / (ph * 0.125)) ** 2)
    gx = np.exp(-0.5 * ((np.arange(pw) - pw / 2) / (pw * 0.125)) ** 2)
    gauss = (gz[:, None, None] * gy[None, :, None] * gx[None, None, :]).astype(np.float32)

    z_pos = list(range(0, max(1, D - pd + 1), sd))
    if D > pd and (D - pd) not in z_pos:
        z_pos.append(D - pd)
    y_pos = list(range(0, max(1, H - ph + 1), sh))
    if H > ph and (H - ph) not in y_pos:
        y_pos.append(H - ph)
    x_pos = list(range(0, max(1, W - pw + 1), sw))
    if W > pw and (W - pw) not in x_pos:
        x_pos.append(W - pw)

    for z in tqdm(z_pos, desc="Inference", leave=False):
        for y in y_pos:
            for x in x_pos:
                patch = volume[z : z + pd, y : y + ph, x : x + pw].astype(np.float32)
                actual_shape = patch.shape

                if patch.shape != (pd, ph, pw):
                    pad = [
                        (0, max(0, pd - patch.shape[0])),
                        (0, max(0, ph - patch.shape[1])),
                        (0, max(0, pw - patch.shape[2])),
                    ]
                    patch = np.pad(patch, pad, mode='reflect')

                patch = normalize_patch_with_stats(patch, vol_mean, vol_std)
                inp = torch.from_numpy(patch[None, None]).to(device)

                with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                    out = model(inp).squeeze()

                out = out.float().cpu().numpy()

                out = out[: actual_shape[0], : actual_shape[1], : actual_shape[2]]
                weight = gauss[: actual_shape[0], : actual_shape[1], : actual_shape[2]]

                logit_sum[z : z + out.shape[0], y : y + out.shape[1], x : x + out.shape[2]] += out * weight
                count[z : z + out.shape[0], y : y + out.shape[1], x : x + out.shape[2]] += weight

    return logit_sum / np.maximum(count, 1e-8)


@torch.no_grad()
def inference_with_tta_logits(
    model,
    volume: np.ndarray,
    patch_size,
    overlap,
    device,
    use_amp=True,
    tta_level='flip',
) -> np.ndarray:
    logits = []
    vol_mean, vol_std = compute_volume_stats(volume)

    # Original
    logits.append(
        sliding_window_inference_logits(model, volume, patch_size, overlap, device, use_amp, vol_mean, vol_std)
    )

    if tta_level in ['flip', 'full']:
        for axis in [0, 1, 2]:
            vol_flip = np.flip(volume, axis).copy()
            log_flip = sliding_window_inference_logits(model, vol_flip, patch_size, overlap, device, use_amp, vol_mean, vol_std)
            logits.append(np.flip(log_flip, axis).copy())

    if tta_level == 'full':
        vol_rot = np.rot90(volume, k=1, axes=(1, 2)).copy()
        log_rot = sliding_window_inference_logits(model, vol_rot, patch_size, overlap, device, use_amp, vol_mean, vol_std)
        logits.append(np.rot90(log_rot, k=-1, axes=(1, 2)).copy())

    return np.mean(logits, axis=0)


# =============================================================================
# CHECKPOINT LOADING / AVERAGING
# =============================================================================

def load_model(checkpoint_path: Path) -> nn.Module:
    model = ResEncUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
    )

    checkpoint = torch.load(checkpoint_path, map_location=cfg.DEVICE, weights_only=False)
    state_dict = checkpoint.get('model_state_dict', checkpoint)

    cleaned = {}
    for k, v in state_dict.items():
        key = k.replace('module.', '').replace('_orig_mod.', '')
        if key in model.state_dict():
            cleaned[key] = v

    model.load_state_dict(cleaned, strict=False)
    return model.to(cfg.DEVICE).eval()


def _epoch_num_from_path(p: str) -> int:
    m = re.search(r"checkpoint_epoch_(\\d+)", p)
    return int(m.group(1)) if m else -1


def average_checkpoints_to_file(glob_pattern: str, last_n: int, out_path: Path) -> Path:
    paths = glob.glob(glob_pattern)
    if len(paths) == 0:
        raise FileNotFoundError(f"No checkpoints matched: {glob_pattern}")

    paths = sorted(paths, key=_epoch_num_from_path)
    selected = paths[-int(last_n):]
    print("Averaging checkpoints:")
    for p in selected:
        print(" ", p)

    avg = None
    n = 0
    for p in selected:
        ck = torch.load(p, map_location='cpu', weights_only=False)
        sd = ck.get('model_state_dict', ck)

        cleaned = {}
        for k, v in sd.items():
            k2 = k.replace('module.', '').replace('_orig_mod.', '')
            if torch.is_tensor(v):
                cleaned[k2] = v.float()

        if avg is None:
            avg = {k: v.clone() for k, v in cleaned.items()}
        else:
            for k in avg.keys():
                if k in cleaned:
                    avg[k] += cleaned[k]
        n += 1

    for k in avg.keys():
        avg[k] /= n

    out_path = Path(out_path)
    torch.save({'model_state_dict': avg}, out_path)
    print("Saved averaged checkpoint:", out_path)
    return out_path


# =============================================================================
# ENSEMBLE PREDICTION (LOGITS)
# =============================================================================

def predict_volume_ensemble_logits(
    volume: np.ndarray,
    checkpoint_paths: List[Path],
    model_weights: Optional[List[float]] = None,
    patch_size=(192, 192, 192),
    overlap=0.7,
    tta_level='flip',
) -> np.ndarray:
    if model_weights is None:
        model_weights = [1.0] * len(checkpoint_paths)
    assert len(model_weights) == len(checkpoint_paths)

    log_sum = None
    w_sum = 0.0

    for ckpt_path, w in zip(checkpoint_paths, model_weights):
        ckpt_path = Path(ckpt_path)
        print("Loading model:", ckpt_path)
        model = load_model(ckpt_path)

        logits = inference_with_tta_logits(
            model,
            volume,
            patch_size=patch_size,
            overlap=overlap,
            device=cfg.DEVICE,
            use_amp=cfg.USE_AMP,
            tta_level=tta_level,
        )

        if log_sum is None:
            log_sum = logits * float(w)
        else:
            log_sum += logits * float(w)

        w_sum += float(w)

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return log_sum / max(w_sum, 1e-8)


# =============================================================================
# VALIDATION SWEEP (scroll 26002)
# =============================================================================

def make_scroll_val_ids() -> List[str]:
    df = pd.read_csv(cfg.TRAIN_CSV)
    ids = df.loc[
        (df['scroll_id'] == cfg.TARGET_SCROLL_ID) & (df['id'].astype(str) != str(cfg.TEST_ID)),
        'id'
    ].astype(str).tolist()
    ids = sorted(ids)
    rng = np.random.RandomState(cfg.VAL_SEED)
    rng.shuffle(ids)
    val_ids = ids[: min(int(cfg.VAL_N), len(ids))]
    if len(val_ids) == 0:
        raise ValueError(f"No ids found for scroll_id={cfg.TARGET_SCROLL_ID}")
    return val_ids


def run_val_sweep(checkpoint_paths: List[Path], model_weights: Optional[List[float]]):
    val_ids = make_scroll_val_ids()
    print("Val IDs:", val_ids)

    results = []

    for overlap in cfg.OVERLAP_SWEEP:
        for tta in cfg.TTA_SWEEP:
            print("\n" + "=" * 70)
            print(f"[SWEEP] overlap={overlap} tta={tta}")
            print("=" * 70)

            prob_maps = {}
            labels = {}
            ignores = {}

            for vid in val_ids:
                vpath = cfg.TRAIN_IMAGES_DIR / f"{vid}.tif"
                lpath = cfg.TRAIN_LABELS_DIR / f"{vid}.tif"

                vol = tifffile.imread(str(vpath))
                lbl = tifffile.imread(str(lpath))

                logits = predict_volume_ensemble_logits(
                    vol,
                    checkpoint_paths=checkpoint_paths,
                    model_weights=model_weights,
                    patch_size=cfg.PATCH_SIZE,
                    overlap=overlap,
                    tta_level=tta,
                )

                logits = smooth_logits_z(logits)
                prob = sigmoid_np(logits)

                prob_maps[vid] = prob
                labels[vid] = (lbl == 1)
                ignores[vid] = (lbl == 2)

                del vol, lbl, logits
                gc.collect()

            for thr in cfg.THRESH_SWEEP:
                for min_comp in cfg.MIN_COMPONENT_SWEEP:
                    for keep_k in cfg.KEEP_LARGEST_SWEEP:
                        dice_list = []
                        for vid in val_ids:
                            pred = postprocess(prob_maps[vid], threshold=thr, min_component_size=min_comp, keep_largest_k=keep_k)
                            d = dice_score(pred, labels[vid], ignore=ignores[vid])
                            dice_list.append(d)
                        mean_dice = float(np.mean(dice_list))
                        results.append({
                            'overlap': overlap,
                            'tta': tta,
                            'thr': thr,
                            'min_comp': min_comp,
                            'keep_k': keep_k,
                            'dice': mean_dice,
                        })
                        print(f"dice={mean_dice:.4f} | thr={thr} min={min_comp} keep_k={keep_k}")

            # free between sweep configs
            del prob_maps, labels, ignores
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    results = sorted(results, key=lambda x: x['dice'], reverse=True)
    print("\nTop 10 configs:")
    for r in results[:10]:
        print(r)

    return results


# =============================================================================
# MAIN
# =============================================================================

def main():
    checkpoint_paths = list(cfg.CHECKPOINT_PATHS)
    model_weights = cfg.MODEL_WEIGHTS

    if cfg.BUILD_AVG_CHECKPOINT:
        avg_path = average_checkpoints_to_file(cfg.CKPT_GLOB, cfg.AVG_LAST_N, cfg.AVG_OUT_PATH)
        checkpoint_paths.append(avg_path)
        if model_weights is not None:
            model_weights = list(model_weights) + [1.0]

    print("\n" + "=" * 70)
    print("INFERENCE CONFIG")
    print("=" * 70)
    print("Checkpoints:")
    for p in checkpoint_paths:
        print(" ", p)
    print(f"Weights: {model_weights}")
    print(f"Overlap: {cfg.OVERLAP}")
    print(f"TTA: {cfg.TTA_LEVEL}")
    print(f"Z smoothing: {cfg.Z_SMOOTH} kernel={cfg.Z_KERNEL}")
    print(f"Threshold: {cfg.FINAL_THRESHOLD}")
    print(f"Min component: {cfg.MIN_COMPONENT_SIZE}")
    print(f"Keep largest K: {cfg.KEEP_LARGEST_K}")

    if cfg.RUN_VAL_SWEEP:
        run_val_sweep(checkpoint_paths, model_weights)
        return

    test_df = pd.read_csv(cfg.TEST_CSV)
    print(f"Found {len(test_df)} test volumes")

    with zipfile.ZipFile(cfg.SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for idx, row in test_df.iterrows():
            image_id = str(row["id"])
            tif_path = cfg.TEST_IMAGES_DIR / f"{image_id}.tif"

            print(f"\n[{idx+1}/{len(test_df)}] {image_id}")
            volume = tifffile.imread(str(tif_path))

            logits = predict_volume_ensemble_logits(
                volume,
                checkpoint_paths=checkpoint_paths,
                model_weights=model_weights,
                patch_size=cfg.PATCH_SIZE,
                overlap=cfg.OVERLAP,
                tta_level=cfg.TTA_LEVEL,
            )

            logits = smooth_logits_z(logits)
            prob = sigmoid_np(logits)

            print(f"Prob range: [{prob.min():.4f}, {prob.max():.4f}] mean={prob.mean():.4f}")

            pred = postprocess(
                prob,
                threshold=cfg.FINAL_THRESHOLD,
                min_component_size=cfg.MIN_COMPONENT_SIZE,
                keep_largest_k=cfg.KEEP_LARGEST_K,
            )

            fg = 100.0 * pred.mean()
            print(f"FG% after postprocess: {fg:.3f}%")

            out_path = cfg.OUTPUT_DIR / f"{image_id}.tif"
            tifffile.imwrite(out_path, pred.astype(np.uint8))
            zf.write(out_path, arcname=f"{image_id}.tif")
            out_path.unlink()

            del volume, logits, prob, pred
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print("\n" + "=" * 70)
    print("Done:", cfg.SUBMISSION_ZIP)
    print("=" * 70)


if __name__ == "__main__":
    main()
